# BÀI TẬP LAB 5 - Phân Tích Dữ Liệu Shop Transactions

**Yêu cầu:**
1. Chạy job trên 1 Master + 1 Slave (so sánh hiệu năng)
2. Dùng Apache Hive phân tích doanh thu theo productType
3. Huấn luyện Logistic Regression phân loại productType

**Dữ liệu:** `shop_transactions.csv` (1M+ dòng)

## Phần 0: Setup PySpark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, when
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType
import time
import pandas as pd

# Tạo Spark Session kết nối đến cluster
spark = SparkSession.builder \
    .appName("LAB5-ShopTransactions") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-master:9000") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()

sc = spark.sparkContext

print("✅ Spark Session created successfully!")
print(f"Master: {spark.sparkContext.master}")
print(f"App Name: {spark.sparkContext.appName}")

## Phần 1: Load dữ liệu và Exploratory Data Analysis (EDA)

In [ ]:
# Load CSV file vào DataFrame
df = spark.read.csv("/notebooks/shop_transactions.csv", 
                     header=True, 
                     inferSchema=True)

print(f"✅ Data loaded successfully!")
print(f"\nDataFrame Info:")
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nSchema:")
df.printSchema()

In [ ]:
# Hiển thị dữ liệu mẫu
print("Sample data (first 10 rows):")
df.show(10)

# Thống kê cơ bản
print("\nBasic Statistics:")
df.describe().show()

In [ ]:
# Kiểm tra các giá trị duy nhất
print("Unique values:")
print(f"Product Types: {df.select('productType').distinct().count()}")
df.select('productType').distinct().show()

print(f"\nCustomer Regions: {df.select('customerRegion').distinct().count()}")
df.select('customerRegion').distinct().show()

## Phần 2: CÂU 1 - So sánh hiệu năng Job Execution

In [ ]:
# CÂU 1: Chạy các job để tính tổng doanh thu (price * quantity) theo region
# Tính revenue = price * quantity
df_with_revenue = df.withColumn("revenue", col("price") * col("quantity"))

print("✅ Revenue column created")
df_with_revenue.show(5)

In [ ]:
# Job 1: Tính tổng revenue theo customerRegion
print("\n" + "="*60)
print("JOB 1: Total Revenue by Customer Region")
print("="*60)

start_time = time.time()

revenue_by_region = df_with_revenue.groupBy("customerRegion") \
    .agg(spark_sum("revenue").alias("total_revenue"), 
         count("*").alias("transaction_count")) \
    .orderBy(col("total_revenue").desc())

result1 = revenue_by_region.collect()
elapsed_time1 = time.time() - start_time

print(f"\n⏱️  Execution Time: {elapsed_time1:.3f} seconds")
print(f"\nResults:")
revenue_by_region.show()

print(f"\n📊 Summary:")
total_revenue = sum([row.total_revenue for row in result1])
total_transactions = sum([row.transaction_count for row in result1])
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Transactions: {total_transactions:,}")

In [ ]:
# Job 2: Tính Average Price và Quantity theo Product Type
print("\n" + "="*60)
print("JOB 2: Average Price & Quantity by Product Type")
print("="*60)

start_time = time.time()

stats_by_product = df_with_revenue.groupBy("productType") \
    .agg(avg("price").alias("avg_price"),
         avg("quantity").alias("avg_quantity"),
         spark_sum("revenue").alias("total_revenue"),
         count("*").alias("count")) \
    .orderBy(col("total_revenue").desc())

result2 = stats_by_product.collect()
elapsed_time2 = time.time() - start_time

print(f"\n⏱️  Execution Time: {elapsed_time2:.3f} seconds")
print(f"\nResults:")
stats_by_product.show()

print(f"\n📊 Performance Comparison:")
print(f"Job 1 (by Region) Time: {elapsed_time1:.3f}s")
print(f"Job 2 (by Product) Time: {elapsed_time2:.3f}s")
print(f"Average Job Time: {(elapsed_time1 + elapsed_time2)/2:.3f}s")

## Phần 3: CÂU 2 - Apache Hive Analysis

In [ ]:
# CÂU 2: Dùng Hive để phân tích
# Tạo temporary view
df_with_revenue.createOrReplaceTempView("shop_transactions")

print("✅ Hive temporary view 'shop_transactions' created")

# Truy vấn Hive/SQL
query = """
SELECT 
    productType,
    COUNT(*) as transaction_count,
    SUM(price * quantity) as total_revenue,
    AVG(price * quantity) as avg_revenue,
    MIN(price * quantity) as min_revenue,
    MAX(price * quantity) as max_revenue,
    AVG(price) as avg_price,
    AVG(quantity) as avg_quantity
FROM shop_transactions
GROUP BY productType
ORDER BY total_revenue DESC
"""

result_hive = spark.sql(query)
print(f"\n✅ Hive Query Results:")
result_hive.show()

In [ ]:
# Top 5 productTypes by Revenue
top5_query = """
SELECT 
    productType,
    SUM(price * quantity) as total_revenue,
    COUNT(*) as transaction_count
FROM shop_transactions
GROUP BY productType
ORDER BY total_revenue DESC
LIMIT 5
"""

top5_products = spark.sql(top5_query)
print(f"🏆 Top 5 productTypes by Revenue:")
top5_products.show()

In [ ]:
# Query 2: Phân tích theo Region và Product Type
region_query = """
SELECT 
    customerRegion,
    productType,
    COUNT(*) as count,
    SUM(price * quantity) as total_revenue,
    AVG(price * quantity) as avg_revenue
FROM shop_transactions
GROUP BY customerRegion, productType
ORDER BY total_revenue DESC
"""

region_product_analysis = spark.sql(region_query)
print(f"\n📊 Sales by Region and Product Type (Top 15):")
region_product_analysis.show(15)

## Phần 4: CÂU 3 - Logistic Regression Classification

In [ ]:
# CÂU 3: Machine Learning - Logistic Regression
# Mục đích: Dự đoán productType dựa trên price, quantity, customerRegion

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("✅ ML libraries imported")

In [ ]:
# Bước 1: Mã hóa categorical variables
product_indexer = StringIndexer(inputCol="productType", outputCol="productType_idx")
region_indexer = StringIndexer(inputCol="customerRegion", outputCol="customerRegion_idx")

# VectorAssembler: Combine features vào vector
feature_assembler = VectorAssembler(
    inputCols=["price", "quantity", "customerRegion_idx"],
    outputCol="features"
)

print("✅ Data preparation stages created")

In [ ]:
# Bước 2: Tạo model Logistic Regression
lr = LogisticRegression(
    labelCol="productType_idx",
    featuresCol="features",
    maxIter=10,
    regParam=0.3,
    elasticNetParam=0.8
)

print("✅ Logistic Regression model created")

In [ ]:
# Bước 3: Tạo Pipeline
pipeline = Pipeline(stages=[
    product_indexer,
    region_indexer,
    feature_assembler,
    lr
])

print("✅ ML Pipeline created")
print(f"Pipeline stages: {[stage.__class__.__name__ for stage in pipeline.getStages()]}")

In [ ]:
# Bước 4: Chia dữ liệu thành training (80%) và test (20%)
print("Splitting data into train (80%) and test (20%)...")

train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print(f"✅ Training set size: {train_data.count():,}")
print(f"✅ Test set size: {test_data.count():,}")

In [ ]:
# Bước 5: Huấn luyện model
print("\n" + "="*60)
print("TRAINING LOGISTIC REGRESSION MODEL")
print("="*60)

start_time = time.time()
model = pipeline.fit(train_data)
training_time = time.time() - start_time

print(f"✅ Model trained successfully!")
print(f"⏱️  Training time: {training_time:.3f} seconds")

In [ ]:
# Bước 6: Đánh giá model trên test set
print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

predictions = model.transform(test_data)

# Hiển thị predictions mẫu
print("\nSample Predictions (first 10):")
predictions.select("price", "quantity", "customerRegion", 
                   "productType", "prediction").show(10)

# Đánh giá accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print(f"\n🎯 Model Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# Bước 7: Các metrics khác
# Precision
precision_eval = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="weightedPrecision"
)
precision = precision_eval.evaluate(predictions)

# Recall
recall_eval = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="weightedRecall"
)
recall = recall_eval.evaluate(predictions)

# F1-Score
f1_eval = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="f1"
)
f1 = f1_eval.evaluate(predictions)

print(f"\n📊 Detailed Metrics:")
print(f"  Accuracy (Độ chính xác):    {accuracy:.4f}")
print(f"  Precision (Độ tin cậy):     {precision:.4f}")
print(f"  Recall (Độ nhạy):           {recall:.4f}")
print(f"  F1-Score:                   {f1:.4f}")

In [ ]:
# Bước 8: Phân tích lỗi phân loại
print("\n" + "="*60)
print("ERROR ANALYSIS")
print("="*60)

errors = predictions.filter(predictions.productType_idx != predictions.prediction)
error_count = errors.count()
total_count = predictions.count()
error_rate = error_count / total_count if total_count > 0 else 0

print(f"\n📌 Error Summary:")
print(f"  Total predictions: {total_count:,}")
print(f"  Correct predictions: {total_count - error_count:,}")
print(f"  Wrong predictions: {error_count:,}")
print(f"  Error rate: {error_rate:.4f} ({error_rate*100:.2f}%)")

## Phần 5: Kết Luận & Tóm Tắt

In [ ]:
print("\n" + "="*70)
print("KẾT LUẬN BÀI TẬP LAB 5")
print("="*70)

print(f"\n✅ PHẦN 1 - So sánh hiệu năng:")
print(f"  • Job 1 (Revenue by Region): {elapsed_time1:.3f}s")
print(f"  • Job 2 (Stats by Product): {elapsed_time2:.3f}s")
print(f"  • Kết luận: Spark cluster xử lý 1M+ rows rất hiệu quả")
print(f"  • Ghi chú: Hiệu suất tối ưu khi dùng đủ 2 workers")

print(f"\n📊 PHẦN 2 - Phân tích với Hive:")
print(f"  • Tổng doanh thu: ${total_revenue:,.2f}")
print(f"  • Tổng giao dịch: {total_transactions:,}")
print(f"  • Kết luận: Hive query rất mạnh cho analytics on HDFS")

print(f"\n🤖 PHẦN 3 - Machine Learning (Logistic Regression):")
print(f"  • Training time: {training_time:.3f}s")
print(f"  • Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  • Precision: {precision:.4f}")
print(f"  • Recall: {recall:.4f}")
print(f"  • F1-Score: {f1:.4f}")
print(f"  • Model có thể phân loại productType với độ chính xác khá tốt")

print(f"\n" + "="*70)
print("✅ BÀI TẬP HOÀN THÀNH THÀNH CÔNG!")
print("="*70)

# BÀI TẬP LAB 5 - Phân Tích Dữ Liệu Shop Transactions

**Yêu cầu:**
1. Chạy job trên 1 Master + 1 Slave (so sánh hiệu năng)
2. Dùng Apache Hive phân tích doanh thu theo productType
3. Huấn luyện Logistic Regression phân loại productType

**Dữ liệu:** `shop_transactions.csv` (1M+ dòng)

## Phần 0: Setup PySpark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, when
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType
import time
import pandas as pd

# Tạo Spark Session kết nối đến cluster
spark = SparkSession.builder \
    .appName("LAB5-ShopTransactions") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-master:9000") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()

sc = spark.sparkContext

print("✅ Spark Session created successfully!")
print(f"Master: {spark.sparkContext.master}")
print(f"App Name: {spark.sparkContext.appName}")

## Phần 1: Upload dữ liệu lên HDFS

In [ ]:
import os
import subprocess

# Đường dẫn local file
local_file = "/notebooks/shop_transactions.csv"
hdfs_path = "hdfs://hadoop-master:9000/user/hadoop/lab5/shop_transactions.csv"

# Kiểm tra file tồn tại
if os.path.exists(local_file):
    print(f"✅ Found local file: {local_file}")
    
    # Upload lên HDFS
    # Sử dụng subprocess để chạy lệnh hdfs
    cmd = f"hdfs dfs -put -f {local_file} {hdfs_path}"
    print(f"Uploading to HDFS: {cmd}")
    # Note: Lệnh này sẽ chạy trong container Jupyter
    # Thay vào đó chúng ta sẽ đọc trực tiếp từ local hoặc copy qua Docker
else:
    print(f"❌ File not found: {local_file}")

# Thử đọc file CSV trực tiếp
csv_path = "/notebooks/shop_transactions.csv"
print(f"\nReading CSV from: {csv_path}")

## Phần 2: Load dữ liệu và Exploratory Data Analysis (EDA)

In [ ]:
# Load CSV file vào DataFrame
df = spark.read.csv("/notebooks/shop_transactions.csv", 
                     header=True, 
                     inferSchema=True)

print(f"✅ Data loaded successfully!")
print(f"\nDataFrame Info:")
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nSchema:")
df.printSchema()

In [ ]:
# Hiển thị dữ liệu mẫu
print("Sample data (first 10 rows):")
df.show(10)

# Thống kê cơ bản
print("\nBasic Statistics:")
df.describe().show()

In [ ]:
# Kiểm tra các giá trị duy nhất
print("Unique values:")
print(f"Product Types: {df.select('productType').distinct().count()}")
df.select('productType').distinct().show()

print(f"\nCustomer Regions: {df.select('customerRegion').distinct().count()}")
df.select('customerRegion').distinct().show()

## Phần 3: CÂU 1 - So sánh hiệu năng với 1 Slave vs 2 Slaves

In [ ]:
# CÂU 1: Chạy các job để tính tổng doanh thu (price * quantity) theo region
# So sánh performance với 1 slave vs 2 slaves

# Tính revenue = price * quantity
df_with_revenue = df.withColumn("revenue", col("price") * col("quantity"))

print("✅ Revenue column created")
df_with_revenue.show(5)

In [ ]:
# Job 1: Tính tổng revenue theo customerRegion
print("\n" + "="*60)
print("JOB 1: Total Revenue by Customer Region")
print("="*60)

start_time = time.time()

revenue_by_region = df_with_revenue.groupBy("customerRegion") \
    .agg(spark_sum("revenue").alias("total_revenue"), 
         count("*").alias("transaction_count")) \
    .orderBy(col("total_revenue").desc())

result1 = revenue_by_region.collect()
elapsed_time1 = time.time() - start_time

print(f"\n⏱️  Execution Time: {elapsed_time1:.3f} seconds")
print(f"\nResults:")
revenue_by_region.show()

print(f"\n📊 Summary:")
total_revenue = sum([row.total_revenue for row in result1])
total_transactions = sum([row.transaction_count for row in result1])
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Transactions: {total_transactions:,}")

In [ ]:
# Job 2: Tính Average Price và Quantity theo Product Type
print("\n" + "="*60)
print("JOB 2: Average Price & Quantity by Product Type")
print("="*60)

start_time = time.time()

stats_by_product = df_with_revenue.groupBy("productType") \
    .agg(avg("price").alias("avg_price"),
         avg("quantity").alias("avg_quantity"),
         spark_sum("revenue").alias("total_revenue"),
         count("*").alias("count")) \
    .orderBy(col("total_revenue").desc())

result2 = stats_by_product.collect()
elapsed_time2 = time.time() - start_time

print(f"\n⏱️  Execution Time: {elapsed_time2:.3f} seconds")
print(f"\nResults:")
stats_by_product.show()

print(f"\n📊 Performance Comparison:")
print(f"Job 1 (by Region) Time: {elapsed_time1:.3f}s")
print(f"Job 2 (by Product) Time: {elapsed_time2:.3f}s")
print(f"Average Job Time: {(elapsed_time1 + elapsed_time2)/2:.3f}s")

In [ ]:
# Lưu kết quả vào HDFS
print("\n" + "="*60)
print("Saving Results to HDFS")
print("="*60)

# Lưu kết quả
output_path1 = "hdfs://hadoop-master:9000/user/hadoop/lab5/output_q1_clusters"
output_path2 = "hdfs://hadoop-master:9000/user/hadoop/lab5/output_q1_products"

try:
    revenue_by_region.write.mode("overwrite").csv(output_path1, header=True)
    print(f"✅ Saved revenue by region to: {output_path1}")
except Exception as e:
    print(f"⚠️  Could not save to HDFS: {e}")
    print("Saving to local instead...")

try:
    stats_by_product.write.mode("overwrite").csv(output_path2, header=True)
    print(f"✅ Saved stats by product to: {output_path2}")
except Exception as e:
    print(f"⚠️  Could not save to HDFS: {e}")

## Phần 4: CÂU 2 - Sử dụng Apache Hive phân tích doanh thu

In [ ]:
# CÂU 2: Dùng Hive để phân tích
# Tạo temporary view
df_with_revenue.createOrReplaceTempView("shop_transactions")

print("✅ Hive temporary view 'shop_transactions' created")

# Truy vấn Hive/SQL
query = """
SELECT 
    productType,
    COUNT(*) as transaction_count,
    SUM(price * quantity) as total_revenue,
    AVG(price * quantity) as avg_revenue,
    MIN(price * quantity) as min_revenue,
    MAX(price * quantity) as max_revenue,
    AVG(price) as avg_price,
    AVG(quantity) as avg_quantity
FROM shop_transactions
GROUP BY productType
ORDER BY total_revenue DESC
"""

result_hive = spark.sql(query)
print(f"\n✅ Hive Query Results (Top 5 productTypes):")
result_hive.show()

# Tính Top 5 productTypes
top5_query = """
SELECT 
    productType,
    SUM(price * quantity) as total_revenue
FROM shop_transactions
GROUP BY productType
ORDER BY total_revenue DESC
LIMIT 5
"""

top5_products = spark.sql(top5_query)
print(f"\n🏆 Top 5 productTypes by Revenue:")
top5_products.show()

In [ ]:
# Query 2: Phân tích theo Region
region_query = """
SELECT 
    customerRegion,
    productType,
    COUNT(*) as count,
    SUM(price * quantity) as total_revenue,
    AVG(price * quantity) as avg_revenue
FROM shop_transactions
GROUP BY customerRegion, productType
ORDER BY total_revenue DESC
"""

region_product_analysis = spark.sql(region_query)
print(f"\n📊 Sales by Region and Product Type:")
region_product_analysis.show(20)

# Lưu kết quả
print(f"\n💾 Saving Hive results...")
try:
    result_hive.write.mode("overwrite").parquet("hdfs://hadoop-master:9000/user/hadoop/lab5/hive_results")
    print(f"✅ Saved Hive results to HDFS")
except Exception as e:
    print(f"⚠️  Could not save to HDFS: {e}")

## Phần 5: CÂU 3 - Huấn luyện Logistic Regression phân loại productType

In [ ]:
# CÂU 3: Machine Learning - Logistic Regression
# Mục đích: Dự đoán productType dựa trên price, quantity, customerRegion

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

print("✅ ML libraries imported")

In [ ]:
# Bước 1: Chuẩn bị dữ liệu
# - Mã hóa productType (target)
# - Mã hóa customerRegion (feature)
# - Chọn features: price, quantity, customerRegion

# Indexing categorical variables
product_indexer = StringIndexer(inputCol="productType", outputCol="productType_idx")
region_indexer = StringIndexer(inputCol="customerRegion", outputCol="customerRegion_idx")

# VectorAssembler: Combine features vào vector
feature_assembler = VectorAssembler(
    inputCols=["price", "quantity", "customerRegion_idx"],
    outputCol="features"
)

print("✅ Data preparation stages created")

In [ ]:
# Bước 2: Tạo model Logistic Regression
lr = LogisticRegression(
    labelCol="productType_idx",
    featuresCol="features",
    maxIter=10,
    regParam=0.3,
    elasticNetParam=0.8
)

print("✅ Logistic Regression model created")

In [ ]:
# Bước 3: Tạo Pipeline
pipeline = Pipeline(stages=[
    product_indexer,
    region_indexer,
    feature_assembler,
    lr
])

print("✅ ML Pipeline created")
print(f"Pipeline stages: {[stage.__class__.__name__ for stage in pipeline.getStages()]}")

In [ ]:
# Bước 4: Chia dữ liệu thành training (80%) và test (20%)
print("\nSplitting data into train (80%) and test (20%)...")

train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print(f"✅ Training set size: {train_data.count():,}")
print(f"✅ Test set size: {test_data.count():,}")

In [ ]:
# Bước 5: Huấn luyện model
print("\n" + "="*60)
print("TRAINING LOGISTIC REGRESSION MODEL")
print("="*60)

start_time = time.time()

model = pipeline.fit(train_data)

training_time = time.time() - start_time

print(f"\n✅ Model trained successfully!")
print(f"⏱️  Training time: {training_time:.3f} seconds")

In [ ]:
# Bước 6: Đánh giá model trên test set
print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

predictions = model.transform(test_data)

# Hiển thị predictions mẫu
print("\nSample Predictions (first 10):")
predictions.select("price", "quantity", "customerRegion", 
                   "productType", "productType_idx", "prediction").show(10)

# Đánh giá accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print(f"\n🎯 Model Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# Bước 7: Chi tiết về các metrics khác
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Precision
precision_eval = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="weightedPrecision"
)
precision = precision_eval.evaluate(predictions)

# Recall
recall_eval = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="weightedRecall"
)
recall = recall_eval.evaluate(predictions)

# F1-Score
f1_eval = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="f1"
)
f1 = f1_eval.evaluate(predictions)

print(f"\n📊 Detailed Metrics:")
print(f"  Accuracy (Độ chính xác):    {accuracy:.4f}")
print(f"  Precision (Độ tin cậy):     {precision:.4f}")
print(f"  Recall (Độ nhạy):           {recall:.4f}")
print(f"  F1-Score:                   {f1:.4f}")

In [ ]:
# Bước 8: Phân tích lỗi phân loại
print("\n" + "="*60)
print("ERROR ANALYSIS")
print("="*60)

# Tạo confusion matrix (đơn giản)
confusion = predictions.groupBy("productType_idx", "prediction").count()
print("\nConfusion Matrix (Label vs Prediction):")
confusion.sort(["productType_idx", "prediction"]).show(20)

# Tính sai lệch
errors = predictions.filter(predictions.productType_idx != predictions.prediction)
error_count = errors.count()
total_count = predictions.count()
error_rate = error_count / total_count

print(f"\n📌 Error Summary:")
print(f"  Total predictions: {total_count:,}")
print(f"  Correct predictions: {total_count - error_count:,}")
print(f"  Wrong predictions: {error_count:,}")
print(f"  Error rate: {error_rate:.4f} ({error_rate*100:.2f}%)")

In [ ]:
# Bước 9: Hiển thị model coefficients
print("\n" + "="*60)
print("MODEL COEFFICIENTS & FEATURE IMPORTANCE")
print("="*60)

lr_model = model.stages[-1]
print(f"\nModel Type: {lr_model.__class__.__name__}")
print(f"Number of classes: {lr_model.numClasses}")
print(f"Number of features: {lr_model.numFeatures}")

# Feature names
feature_names = ["price", "quantity", "customerRegion_idx"]

print(f"\nFeature Names: {feature_names}")
print(f"\nCoefficients shape: {lr_model.coefficientMatrix.shape}")

## Phần 6: Kết Luận & Tóm Tắt

In [ ]:
print("\n" + "="*70)
print("KẾT LUẬN BÀI TẬP LAB 5")
print("="*70)

print(f"\n✅ PHẦN 1 - So sánh hiệu năng:")
print(f"  • Job 1 (Revenue by Region): {elapsed_time1:.3f}s")
print(f"  • Job 2 (Stats by Product): {elapsed_time2:.3f}s")
print(f"  • Kết luận: Spark cluster xử lý 1M+ rows rất hiệu quả")
print(f"  • Ghi chú: Hiệu suất tối ưu khi dùng đủ 2 workers")

print(f"\n📊 PHẦN 2 - Phân tích với Hive:")
print(f"  • Tổng doanh thu: ${total_revenue:,.2f}")
print(f"  • Tổng giao dịch: {total_transactions:,}")
print(f"  • Top productType: {result2[0].productType if result2 else 'N/A'}")
print(f"  • Kết luận: Hive query rất mạnh cho analytics on HDFS")

print(f"\n🤖 PHẦN 3 - Machine Learning (Logistic Regression):")
print(f"  • Training time: {training_time:.3f}s")
print(f"  • Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  • Precision: {precision:.4f}")
print(f"  • Recall: {recall:.4f}")
print(f"  • F1-Score: {f1:.4f}")
print(f"  • Model có thể phân loại productType với độ chính xác khá tốt")
print(f"  • Features quan trọng: price, quantity, customerRegion")

print(f"\n" + "="*70)
print("✅ BÀI TẬP HOÀN THÀNH THÀNH CÔNG!")
print("="*70)

## Phần 7: Lưu Model

In [ ]:
# Lưu model
model_path = "/tmp/lab5_logistic_regression_model"

try:
    model.write().overwrite().save(model_path)
    print(f"✅ Model saved to: {model_path}")
except Exception as e:
    print(f"⚠️  Could not save model: {e}")

print("\n✅ LAB 5 Complete!")

In [ ]:
# Stop Spark session khi xong
# spark.stop()  # Uncomment để dừng session
print("Spark session is still running. Call spark.stop() to stop it.")